# 09: Covariance eigenspectra and effective rank

![A feature cloud, its eigenvalue spectrum, and the effective rank computed from that spectrum](../images/09_eigenspectra_and_effective_rank.svg)

A representation with 512 columns may still vary along only a handful of directions. This notebook measures that directly. We build data whose true dimensionality we know, recover it two different ways, and then summarize the recovered spectrum with one number.

**Learning goals:** connect covariance eigenvalues to projected variance and to singular values, implement PCA, handle tied eigenspaces, and compute entropy effective rank. Read the full argument in the [lesson 09 lecture](../lectures/09_eigenspectra_and_effective_rank.md).

**Curriculum role:** this is an optional representation diagnostic, not a primary estimand or decision gate in the hierarchical-diversity experiment.

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

SEED = 9
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
rng = np.random.default_rng(SEED)
print(f'NumPy {np.__version__}, PyTorch {torch.__version__}, device=cpu')

## 1. Create data with known latent dimensionality

To check that a measurement works, start from a case where you already know the answer. We draw three latent variables with variances 9, 4, and 1, rotate them into ten observed coordinates with an orthonormal basis, and add small isotropic noise.

Because the rotation preserves variance and the noise is tiny, the covariance of the ten observed features should have three clearly dominant eigenvalues followed by seven small ones. The assertion below only checks symmetry; the interesting check comes in the next cell.

In [ ]:
N, D, K = 600, 10, 3
basis, _ = np.linalg.qr(rng.normal(size=(D, K)))
latent = rng.normal(size=(N, K)) * np.array([3.0, 2.0, 1.0])
features = latent @ basis.T + 0.15 * rng.normal(size=(N, D))
centered = features - features.mean(axis=0, keepdims=True)
cov = centered.T @ centered / (N - 1)
assert np.allclose(cov, cov.T, atol=1e-12)
print('shape:', features.shape, 'total variance:', round(np.trace(cov), 3))

## 2. Eigendecomposition and SVD agree

The lecture derives the same principal directions twice, once from the covariance matrix and once from the data matrix. Here we run both and confirm they land in the same place.

`np.linalg.eigh` exploits the symmetry of covariance and returns eigenvalues in ascending order, so we reverse them into the descending convention. `np.linalg.svd(..., full_matrices=False)` avoids computing square factors we would never use. For centred data the two routes are linked by `singular_value**2 / (N - 1)`, which is exactly what the assertion tests.

In [ ]:
values, vectors = np.linalg.eigh(cov)
order = np.argsort(values)[::-1]
values, vectors = values[order], vectors[:, order]
_, singular_values, right_vectors_t = np.linalg.svd(centered, full_matrices=False)
svd_values = singular_values**2 / (N - 1)
print('leading eigenvalues:', np.round(values[:5], 3))
print('leading SVD values: ', np.round(svd_values[:5], 3))
assert np.allclose(values, svd_values, rtol=1e-10, atol=1e-10)
assert np.all(values >= -1e-12)

## 3. PCA projection and reconstruction

Now use the directions rather than just inspect them. Multiplying the centred data by the first `K` eigenvectors rewrites every observation in principal coordinates, and multiplying back by the transposed basis returns to the original ten features.

Two properties from the lecture become assertions here. The score covariance is diagonal, so the new coordinates carry no linear redundancy. And the reconstruction keeps almost all the variance, because the discarded directions held almost none of it.

In [ ]:
principal_basis = vectors[:, :K]
scores = centered @ principal_basis
score_cov = scores.T @ scores / (N - 1)
reconstruction = scores @ principal_basis.T
mse = np.mean((centered - reconstruction) ** 2)
explained = values[:K].sum() / values.sum()
print(f'explained variance={explained:.3%}, reconstruction MSE={mse:.4f}')
assert np.allclose(score_cov, np.diag(values[:K]), atol=1e-10)
assert explained > 0.97

## 4. Effective rank summarizes spectral concentration

The spectrum is a list of ten numbers. Effective rank compresses it into one: normalize the nonnegative eigenvalues into a probability distribution, take its Shannon entropy, then exponentiate so the answer reads on a dimension-like scale.

An all-zero spectrum has no normalized variance distribution at all, so the degenerate case needs a policy that is declared rather than assumed. The function below declares that policy as returning `0.0`. The lecture's reference implementation declares it differently, returning `NaN` as an explicit missing-value sentinel. Either is defensible; what matters is that the choice is written down, since neither value follows from the entropy formula.

In [ ]:
def effective_rank(eigenvalues, eps=1e-12):
    nonnegative = np.clip(np.asarray(eigenvalues, dtype=float), 0.0, None)
    total = nonnegative.sum()
    if total <= eps:
        return 0.0
    p = nonnegative / total
    p = p[p > 0]
    entropy = -(p * np.log(p)).sum()
    return float(np.exp(entropy))

print(f'algebraic rank={np.linalg.matrix_rank(cov)}, effective rank={effective_rank(values):.2f}')
assert np.isclose(effective_rank([2, 2, 2, 2]), 4.0)
assert np.isclose(effective_rank([8, 0, 0]), 1.0)

## 5. Tied eigenvalues identify a subspace, not unique axes

When two eigenvalues are exactly equal, any rotation of their two eigenvectors within the shared plane is an equally valid answer. That means individual axes are not identifiable, even though the plane they span is.

The cell below demonstrates the point concretely: rotating an orthonormal basis changes its columns a lot while leaving the projector `Q @ Q.T` unchanged to machine precision. Compare projectors when the scientific object is the subspace, and compare axes only when a deterministic-coordinate contract requires it.

In [ ]:
q, _ = np.linalg.qr(rng.normal(size=(5, 2)))
angle = 0.73
rotation = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
q_rotated = q @ rotation
basis_difference = np.linalg.norm(q - q_rotated)
projector_difference = np.linalg.norm(q @ q.T - q_rotated @ q_rotated.T)
print(f'basis difference={basis_difference:.3f}, projector difference={projector_difference:.2e}')
assert basis_difference > 0.5 and projector_difference < 1e-12

## 6. Canonicalize tied eigenspaces when coordinates must persist

Section 5 showed that a tied eigenspace identifies a projector, not unique axes. Serialization sometimes still needs deterministic coordinates, so this section builds a convention that always picks the same basis.

The recipe follows the lecture: project the ordinary coordinate axes through the projector, orthogonalize the survivors with two passes of modified Gram-Schmidt, and fix every retained vector's sign with an explicit pivot rule. The assertions check the property that matters, which is that two arbitrarily rotated inputs produce the same canonical output. The convention is reproducible, and it does not make the axes scientifically unique.

The last few lines contrast full and thin SVD, because a normalizer that promises a complete coordinate basis, including a null space when the feature count exceeds the sample count, must request `full_matrices=True`.

In [ ]:
def canonical_sign(vector):
    vector = np.asarray(vector, dtype=np.float64)
    magnitudes = np.abs(vector)
    pivot = int(np.flatnonzero(magnitudes == magnitudes.max())[0])
    return -vector if vector[pivot] < 0 else vector

def modified_gram_schmidt(candidate, basis):
    value = np.array(candidate, dtype=np.float64, copy=True)
    for _ in range(2):
        for vector in basis:
            value -= np.dot(vector, value) * vector
    return value

def canonicalize_tied_group(group):
    group = np.asarray(group, dtype=np.float64)
    projector = group.T @ group
    projector = (projector + projector.T) * 0.5
    basis = []
    threshold = 64.0 * np.finfo(np.float64).eps
    for axis in range(projector.shape[1]):
        candidate = modified_gram_schmidt(projector[:, axis], basis)
        norm = np.linalg.norm(candidate)
        if norm > threshold:
            basis.append(canonical_sign(candidate / norm))
        if len(basis) == group.shape[0]:
            break
    result = np.stack(basis)
    assert np.allclose(result @ result.T, np.eye(group.shape[0]), rtol=0.0, atol=1e-12)
    return result

# Rows hold basis vectors. An arbitrary rotation changes rows but not the projector.
base_rows = q.T
rotated_rows = q_rotated.T
canonical_a = canonicalize_tied_group(base_rows)
canonical_b = canonicalize_tied_group(rotated_rows)
assert np.allclose(base_rows.T @ base_rows, rotated_rows.T @ rotated_rows, atol=1e-12)
assert np.allclose(canonical_a, canonical_b, rtol=0.0, atol=1e-12)

wide = rng.normal(size=(4, 7))
wide -= wide.mean(axis=0, keepdims=True)
_, _, thin_vh = np.linalg.svd(wide, full_matrices=False)
_, _, full_vh = np.linalg.svd(wide, full_matrices=True)
assert thin_vh.shape == (4, 7) and full_vh.shape == (7, 7)
assert np.allclose(full_vh @ full_vh.T, np.eye(7), atol=1e-12)
print("canonical tied basis:", canonical_a.shape, "full versus thin:", full_vh.shape, thin_vh.shape)

In [ ]:
torch_x = torch.tensor(centered, dtype=torch.float64)
torch_cov = torch_x.T @ torch_x / (N - 1)
torch_values = torch.linalg.eigvalsh(torch_cov).flip(0).numpy()
assert np.allclose(torch_values, values, atol=1e-10)

fig, ax = plt.subplots(figsize=(7, 3), constrained_layout=True)
ax.bar(np.arange(1, D + 1), values, color=['#c75b4b'] * 3 + ['#8fc5e8'] * (D - 3))
ax.set(xlabel='component', ylabel='eigenvalue', title=f'Spectrum, effective rank = {effective_rank(values):.2f}')
plt.show()

## Exercises and takeaways

Each exercise below changes one input and names the effect you should observe.

1. Omit centering and add a large constant mean. **Check:** the leading singular direction is dominated by that mean.
2. Compare `[9, 1]` and `[5, 5]`. **Check:** both have total variance 10, but the second has larger effective rank.
3. Rotate `base_rows` by several orthogonal matrices. **Check:** the raw axes change while the canonicalized basis remains fixed.
4. Change the canonical sign pivot rule. **Check:** the fitted coordinates may change even though the projector does not.
5. Replace float64 with float32. **Check:** leading values remain stable while the smallest tail values can move more.

**Efficiency:** use symmetric `eigh` for covariance, thin SVD when forming a large covariance is wasteful, and low-rank methods when only leading directions matter.

**Takeaways:** eigenvalues are directional variances, SVD and covariance eigendecomposition agree, PCA gives the optimal linear low-rank reconstruction, tied axes are not individually identifiable, and effective rank measures continuous spectral occupancy. Canonical axes are a deterministic software convention used only when persistent coordinates are required.

## Continue learning

Lesson 10 takes this feature geometry and uses it to fit stable linear predictors.

[Previous notebook: 08](08_group_aware_sampling.ipynb) | [Lecture](../lectures/09_eigenspectra_and_effective_rank.md) | [Curriculum](../README.md) | [Next notebook: 10](10_regularized_linear_estimation.ipynb)